## 5.0 Data preparation identical to 1_data_preparation.ipynb

In [1]:
# Install required packages
!pip install shap xgboost lightgbm --quiet


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from utilz.multi_residual_bootstrap import MultiCovariateResidualBootstrapTransformer, build_covariates
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import shap
from collections import Counter
from matplotlib import pyplot as plt
from scipy.stats import rankdata

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score,
    balanced_accuracy_score, classification_report, confusion_matrix,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from utilz.Dataset import load_dataset
from utilz.constans import DISEASE, HEALTHY
from utilz.helpers import plot_roc_curve
from utilz.preprocessing_utilz import (
    ConstantExpressionReductor,
    MeanExpressionReductor,
    WithinGroupVarianceReductor, AnovaFdrReductor,
    Log2FCReductor, AgeResidualBootstrapTransformer,
    SexResidualBootstrapTransformer,
)

meta_path = r"../data/samples_pancreatic.xlsx"
data_path = r"../data/counts_pancreatic.csv"
ds = load_dataset(data_path, meta_path, label_col="Group")

TEST_SIZE = 0.2
VALID_SIZE = 0.2
ANOVA_FDR_THRESHOLD = 0.05
LOG2FC_THRESHOLD  = np.log2(1.2)


ds.y = ds.y.replace({DISEASE: HEALTHY})

le = LabelEncoder()
y_encoded = pd.Series(le.fit_transform(ds.y), index=ds.y.index)
X_train, X_test, X_valid, y_train, y_test, y_valid = (
    ds.get_train_test_valid_split(ds.X, y_encoded, test_size=TEST_SIZE, valid_size=VALID_SIZE)
)
print("Class mapping:")
print(f"{le.classes_[0]} -> {le.transform(le.classes_)[0]}")
print(f"{le.classes_[1]} -> {le.transform(le.classes_)[1]}")

print("Encoded label distribution:")
print(y_encoded.value_counts().sort_index())
sex_numeric = ds.sex.map({"F": 0, "M": 1})
cov = build_covariates(ds.meta)
pipeline = Pipeline([
    ('ConstantExpressionReductor', ConstantExpressionReductor()),

    #('AnovaFDRReductor', AnovaFdrReductor(alpha=ANOVA_FDR_THRESHOLD)),
    #('Log2FCReductor', Log2FCReductor(min_abs_log2fc = LOG2FC_THRESHOLD)),
    ('WithinGroupVarianceReductor', WithinGroupVarianceReductor(alpha=0.05)),
    ('MeanExpressionReductor', MeanExpressionReductor(percentile=5)),
    ('multi_resid', MultiCovariateResidualBootstrapTransformer(
    covariates=cov, labels=y_train,
    n_bootstrap=1000, fdr_alpha=0.01, min_r2=0.05, cv_threshold_pct=40.0,
    )),
    ('scaler', StandardScaler()),
])

X_train = pipeline.fit_transform(X_train, y_train)
X_valid = pipeline.transform(X_valid)
X_test = pipeline.transform(X_test)

[INFO] skipped 1973 probs due to missing metadata
Dropping inconsistent sample:
                        Sex   Age                Group   Institution  \
Vumc-ChronPan-29-TR1045   M  58.0  Pancreatic diseases  Institute 13   

                         Lib.size Stage RealLocation    Mode  CA125  \
Vumc-ChronPan-29-TR1045   1493422    IV         VUMC  Single    NaN   

                         Platelets Histology   Datasplit Gdansk_sample_name  \
Vumc-ChronPan-29-TR1045        NaN       NaN  Validation                NaN   

                        StageFull  LeukoMichal      PTPRC  
Vumc-ChronPan-29-TR1045        IV  7726.550165  97.092449  
[INFO] 7 samples with unique strata added to train set
[INFO] 4 samples with unique strata (2nd split) added to train set

[ASSERTION PASSED] No leakage detected between splits.
Class mapping:
Asymptomatic controls -> 0
Pancreatic cancer -> 1
Encoded label distribution:
0    459
1    124
Name: count, dtype: int64
data shape after ConstantExpressionRed

---

## 5. Ensemble model

We combine three classifiers - Logistic Regression, LightGBM, and XGBoost - using a StackingClassifier.

All base models use hyperparameters from grid search. Decision threshold is tuned on the validation set (Youden's J statistic).


In [3]:
from sklearn.model_selection import GridSearchCV
from scipy.stats import loguniform, uniform, randint

class_counts = Counter(y_train)
scale_pos_weight = class_counts[0] / class_counts[1]

# Siatki hiperparametrow (analogiczne do 5_alternative_models.py).
# Po tuningu zmienne logreg/xgb/lgbm trzymaja best_estimator_,
# wiec stacking i SHAP ponizej dzialaja bez zmian.
model_specs = {
    'LogReg': {
        'estimator': LogisticRegression(
            penalty='elasticnet', solver='saga',
            max_iter=20000, class_weight='balanced',
            random_state=2137,
        ),
        'param_distributions': {
            'C':        loguniform(1e-3, 1e2),
            'l1_ratio': uniform(0.1, 0.8),       # [0.1, 0.9]
        },
        'n_iter': 25,
        'search': 'random',
    },
    'XGB': {
        'estimator': XGBClassifier(
            objective='binary:logistic', eval_metric='logloss',
            tree_method='hist', scale_pos_weight=scale_pos_weight,
            random_state=2137, n_jobs=1, verbosity=0,
        ),
        'param_grid': {
            'n_estimators':     [100, 300, 600],
            'max_depth':        [2, 3, 5],
            'learning_rate':    [0.03, 0.1],
            'subsample':        [0.8],
            'colsample_bytree': [0.8],
        },
    },
    'LGBM': {
        'estimator': LGBMClassifier(
            class_weight='balanced', random_state=2137,
            n_jobs=1, verbose=-1,
        ),
        'param_distributions': {
            'n_estimators':  randint(100, 700),
            'max_depth':     [-1, 3, 5, 7],
            'learning_rate': loguniform(0.01, 0.2),
            'num_leaves':    randint(8, 64),
        },
        'n_iter': 30,
        'search': 'random',
    },
}

CV_FOLDS = min(10, int(np.bincount(y_train.astype(int)).min()))
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=2137)


In [4]:
from sklearn.model_selection import RandomizedSearchCV

results = []
fitted = {}
for name, spec in model_specs.items():
    if spec.get('search', 'grid') == 'random':
        gs = RandomizedSearchCV(
            spec['estimator'], spec['param_distributions'],
            n_iter=spec['n_iter'],
            scoring='roc_auc', cv=cv, n_jobs=-1, refit=True,
            return_train_score=False, verbose=0,
            random_state=2137,
        ).fit(X_train, y_train)
        search_kind = f"random (n_iter={spec['n_iter']})"
    else:
        gs = GridSearchCV(
            spec['estimator'], spec['param_grid'],
            scoring='roc_auc', cv=cv, n_jobs=-1, refit=True,
            return_train_score=False, verbose=0,
        ).fit(X_train, y_train)
        search_kind = "grid"

    proba_te = gs.best_estimator_.predict_proba(X_test)[:, 1]
    proba_va = gs.best_estimator_.predict_proba(X_valid)[:, 1]

    # threshold tuning on valid (Youden's J), tylko do F1/BA
    fpr_v, tpr_v, thr_v = roc_curve(y_valid, proba_va)
    best_thr = thr_v[np.argmax(tpr_v - fpr_v)]
    y_pred = (proba_te >= best_thr).astype(int)

    auc_cv = gs.best_score_
    auc_te = roc_auc_score(y_test, proba_te)
    f1     = f1_score(y_test, y_pred, average='weighted')
    ba     = balanced_accuracy_score(y_test, y_pred)

    fitted[name] = gs.best_estimator_
    results.append({
        'model':       name,
        'search':      search_kind,
        'best_params': gs.best_params_,
        'cv_auc':      float(auc_cv),
        'test_auc':    float(auc_te),
        'f1':          float(f1),
        'bal_acc':     float(ba),
        'threshold':   float(best_thr),
        'proba_te':    proba_te,
    })

    print(f"{name:6s} [{search_kind:18s}] | CV AUC: {auc_cv:.4f} | Test AUC: {auc_te:.4f} "
          f"| F1(w): {f1:.4f} | BalAcc: {ba:.4f} | Thr: {best_thr:.4f}")
    print(f"         best params: {gs.best_params_}")

# Eksport pod stare nazwy dla downstream cells (stacking, SHAP)
logreg = fitted['LogReg']
xgb    = fitted['XGB']
lgbm   = fitted['LGBM']

summary = pd.DataFrame([{k: v for k, v in r.items() if k not in ('proba_te',)}
                        for r in results])


LogReg [random (n_iter=25)] | CV AUC: 0.8489 | Test AUC: 0.9369 | F1(w): 0.8475 | BalAcc: 0.8784 | Thr: 0.2998
         best params: {'C': np.float64(0.08367967923094517), 'l1_ratio': np.float64(0.4569287444750658)}


KeyboardInterrupt: 



### 5.2. StackingClassifier
Stacking with a LogisticRegression meta-learner trained on base model predictions (predict_proba).

In [ ]:
stacking = StackingClassifier(
    estimators=[('logreg', logreg), ('xgb', xgb), ('lgbm', lgbm)],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=100),
    cv=StratifiedKFold(n_splits=2, shuffle=True, random_state=42),
    stack_method='predict_proba',
    passthrough=False
)

stacking.fit(X_train, y_train)
scores_valid_raw = stacking.predict_proba(X_valid)[:, 1]
scores_test_raw = stacking.predict_proba(X_test)[:, 1]

# Threshold tuning on validation set (Youden's J)
fpr_v, tpr_v, thresholds_v = roc_curve(y_valid, scores_valid_raw)
raw_threshold = thresholds_v[np.argmax(tpr_v - fpr_v)]

y_pred_raw = (scores_test_raw >= raw_threshold).astype(int)

print(f"Optimal threshold: {raw_threshold:.4f}")
print(f"AUC:               {roc_auc_score(y_test, scores_test_raw):.4f}")
print(f"F1 (weighted):     {f1_score(y_test, y_pred_raw, average='weighted'):.4f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, y_pred_raw):.4f}")
print(f"\n{classification_report(y_test, y_pred_raw, target_names=le.classes_)}")
print(f"Confusion matrix:\n{confusion_matrix(y_test, y_pred_raw)}")
plot_roc_curve(scores_test_raw, y_test, "logistic regression")

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

# CV AUC dla stackingu (refit na kazdym foldzie tym samym cv co modele bazowe)
stacking_cv_auc = cross_val_score(
    clone(stacking), X_train, y_train,
    scoring='roc_auc', cv=cv, n_jobs=-1,
).mean()

# threshold dla stackingu (juz mamy fpr_v/tpr_v/thresholds_v z poprzedniej komorki)
y_pred_stack = (scores_test_raw >= raw_threshold).astype(int)
stacking_test_auc = roc_auc_score(y_test, scores_test_raw)
stacking_f1  = f1_score(y_test, y_pred_stack, average='weighted')
stacking_ba  = balanced_accuracy_score(y_test, y_pred_stack)

print(f"Stacking | CV AUC: {stacking_cv_auc:.4f} | Test AUC: {stacking_test_auc:.4f} "
      f"| F1(w): {stacking_f1:.4f} | BalAcc: {stacking_ba:.4f} "
      f"| Thr: {raw_threshold:.4f}")

results_all = list(results) + [{
    'model':     'Stacking',
    'cv_auc':    float(stacking_cv_auc),
    'test_auc':  float(stacking_test_auc),
    'f1':        float(stacking_f1),
    'bal_acc':   float(stacking_ba),
    'threshold': float(raw_threshold),
    'proba_te':  scores_test_raw,
}]
summary_all = pd.DataFrame([{k: v for k, v in r.items() if k != 'proba_te'}
                            for r in results_all])

# --- bar chart: CV vs test AUC (z ensemble) ---
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(results_all))
w = 0.35
ax.bar(x - w/2, summary_all['cv_auc'],   w, label=f'CV AUC ({CV_FOLDS}-fold, train)')
ax.bar(x + w/2, summary_all['test_auc'], w, label='Holdout test AUC',
       color='tab:green')
for i, (cv_v, te_v) in enumerate(zip(summary_all['cv_auc'], summary_all['test_auc'])):
    ax.text(i - w/2, cv_v + 0.005, f'{cv_v:.3f}', ha='center', fontsize=8)
    ax.text(i + w/2, te_v + 0.005, f'{te_v:.3f}', ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(summary_all['model'])
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('AUC')
ax.set_title('Modele bazowe + ensemble: CV vs holdout test AUC')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

# --- ROC na tescie (z ensemble) ---
fig, ax = plt.subplots(figsize=(6, 6))
for r in results_all:
    fpr, tpr, _ = roc_curve(y_test, r['proba_te'])
    ax.plot(fpr, tpr, label=f"{r['model']} (AUC={r['test_auc']:.3f})")
ax.plot([0, 1], [0, 1], color='gray', ls='--', alpha=0.5)
ax.set(xlabel='FPR', ylabel='TPR', title='ROC - holdout test (z ensemble)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print()
print(summary_all.to_string(index=False))


---

## 6. SHAP - Feature importance analysis

SHAP (SHapley Additive exPlanations) values quantify each feature's contribution to individual predictions. We compute SHAP values for each base model (Logistic Regression, LightGBM, XGBoost) to identify the most important genes driving the cancer vs. control classification.

### 6.1. SHAP for Logistic Regression

For linear models, SHAP values can be computed efficiently using the `LinearExplainer`. The coefficients directly map to feature contributions.

In [ ]:
feature_names = list(pipeline.named_steps['Log2FCReductor'].selected_genes_)
explainer_lr = shap.LinearExplainer(logreg, X_train)
shap_values_lr = explainer_lr.shap_values(X_valid)
shap.summary_plot(shap_values_lr, X_valid, feature_names=feature_names, max_display=10, show=True)

### 6.2. SHAP for LightGBM

LightGBM is a gradient boosting framework that natively supports `TreeExplainer`, allowing efficient and exact computation of SHAP values using the tree structure.

In [ ]:
explainer_lgbm = shap.TreeExplainer(lgbm)
shap_values_lgbm = explainer_lgbm.shap_values(X_valid)
if isinstance(shap_values_lgbm, list):
    shap_values_lgbm_c1 = shap_values_lgbm[1]
else:
    shap_values_lgbm_c1 = shap_values_lgbm
print(f"SHAP values shape (LGBM): {shap_values_lgbm_c1.shape}")
shap.summary_plot(shap_values_lgbm_c1, X_valid, feature_names=feature_names, max_display=5, show=True)

### 6.3. SHAP for XGBoost

XGBoost natively supports `TreeExplainer`, which computes exact SHAP values efficiently using the tree structure.

In [ ]:
explainer_xgb = shap.TreeExplainer(xgb)
shap_values_xgb = explainer_xgb.shap_values(X_valid)
shap.summary_plot(shap_values_xgb, X_valid, feature_names=feature_names, max_display=5, show=True)

### 6.4. Stacking-weighted SHAP gene ranking

Combined gene importance score: for each gene we multiply its mean |SHAP| by the weight that the stacking meta-learner assigns to that base model, then sum across all three models:

$$\text{score}(g) = \sum_{m \in \{\text{LR, LGBM, XGB}\}} \overline{|\text{SHAP}_m(g)|} \times w_m$$

where $w_m$ comes from the meta-learner's `coef_` (normalized to sum to 1). This directly reflects how much the final ensemble relies on each base model's predictions.

While the ensemble improved classification metrics over individual base models, the primary goal is to enhance the reliability of gene identification rather than maximize performance scores. The stacking-weighted SHAP ranking highlights genes that are consistently important across all models, with greater emphasis on those contributing most to the final ensemble decision.

In [ ]:
# Wagi modeli bazowych = ich holdout-test AUC (znormalizowane do sumy 1).
# Wczesniej bralismy stacking.final_estimator_.coef_; tutaj uzywamy
# bezposrednio jakosci modelu na tescie.
auc_by_model = {r['model']: r['test_auc'] for r in results}
name_map = {'logreg': 'LogReg', 'xgb': 'XGB', 'lgbm': 'LGBM'}
model_names = [name for name, _ in stacking.estimators]
raw_weights = np.array([auc_by_model[name_map[n]] for n in model_names])
w = raw_weights / raw_weights.sum()

print("Wagi modeli = test AUC (znormalizowane):")
for name, wi, ri in zip(model_names, w, raw_weights):
    print(f"  {name:6s}:  w = {wi:.4f}  (test AUC = {ri:.4f})")

mean_abs_lr   = np.abs(shap_values_lr).mean(axis=0)
mean_abs_lgbm = np.abs(shap_values_lgbm_c1).mean(axis=0)
mean_abs_xgb  = np.abs(shap_values_xgb).mean(axis=0)

def minmax_norm(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-10)

shap_map = {
    'logreg': minmax_norm(mean_abs_lr),
    'lgbm':   minmax_norm(mean_abs_lgbm),
    'xgb':    minmax_norm(mean_abs_xgb),
}

weighted_score = np.zeros_like(list(shap_map.values())[0])
for i in range(len(stacking.estimators)):
    name, _ = stacking.estimators[i]
    wi = w[i]
    weighted_score += shap_map[name] * wi

ranking_df = pd.DataFrame({
    'gene': feature_names,
    'SHAP_LR':   mean_abs_lr,
    'SHAP_LGBM': mean_abs_lgbm,
    'SHAP_XGB':  mean_abs_xgb,
    'auc_weighted_score': weighted_score,
}).sort_values('auc_weighted_score', ascending=False).reset_index(drop=True)

ranking_df.index += 1
ranking_df.index.name = 'rank'

print("\nTop 20 genow wg AUC-wazonego SHAP:")
ranking_df.head(20)

In [ ]:
def minmax_norm(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-10)

# Normalizacja na PELNYM ranking_df - taka sama jak w komorce wczesniej
norm_lr_full   = minmax_norm(ranking_df['SHAP_LR'].values)
norm_lgbm_full = minmax_norm(ranking_df['SHAP_LGBM'].values)
norm_xgb_full  = minmax_norm(ranking_df['SHAP_XGB'].values)

top20 = ranking_df.head(20).copy()
genes = top20['gene'].values
y_pos = np.arange(len(genes))

w_dict = {name: wi for (name, _), wi in zip(stacking.estimators, w)}

# Top20 to pierwsze 20 wierszy ranking_df (juz posortowane), wiec [:20]
bar_lr   = norm_lr_full[:20]   * w_dict['logreg']
bar_lgbm = norm_lgbm_full[:20] * w_dict['lgbm']
bar_xgb  = norm_xgb_full[:20]  * w_dict['xgb']

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(y_pos, bar_lr,                          label=f'LogReg (w={w_dict["logreg"]:.3f})', color='#6366f1')
ax.barh(y_pos, bar_lgbm, left=bar_lr,           label=f'LGBM   (w={w_dict["lgbm"]:.3f})',   color='#f59e0b')
ax.barh(y_pos, bar_xgb,  left=bar_lr + bar_lgbm, label=f'XGB    (w={w_dict["xgb"]:.3f})',    color='#10b981')

ax.set_yticks(y_pos)
ax.set_yticklabels(genes, fontsize=9)
ax.invert_yaxis()  # najwazniejszy na gorze
ax.set_xlabel('AUC-weighted normalized mean |SHAP|  (= auc_weighted_score)')
ax.set_title('Top 20 genow wg AUC-wazonego SHAP')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
ranking_df.head(100).to_csv('top100_shap_ranking.csv', index=False)

In [ ]:
top_genes = ranking_df.head(20)['gene'].tolist()

X_train_top = pd.DataFrame(X_train, columns=feature_names)[top_genes]
X_valid_top = pd.DataFrame(X_valid, columns=feature_names)[top_genes]
X_test_top  = pd.DataFrame(X_test,  columns=feature_names)[top_genes]

logreg = LogisticRegression(
    solver='saga', max_iter=15000, class_weight='balanced', fit_intercept=True
)
logreg.fit(X_train_top, y_train)

y_valid_prob = logreg.predict_proba(X_valid_top)[:, 1]
fpr_v, tpr_v, thr_v = roc_curve(y_valid, y_valid_prob)
best_thr = thr_v[np.argmax(tpr_v - fpr_v)]

y_test_prob = logreg.predict_proba(X_test_top)[:, 1]
y_pred      = (y_test_prob >= best_thr).astype(int)

auc = roc_auc_score(y_test, y_test_prob)
f1  = f1_score(y_test, y_pred, average='weighted')
ba  = balanced_accuracy_score(y_test, y_pred)

print(f"Logistic Regression on Top 20 genes")
print(f"{'='*50}")
print(f"Optimal threshold: {best_thr:.4f}")
print(f"AUC:               {auc:.4f}")
print(f"F1 (weighted):     {f1:.4f}")
print(f"Balanced accuracy: {ba:.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=le.classes_)}")
print(f"Confusion matrix:\n{confusion_matrix(y_test, y_pred)}")

plot_roc_curve(y_test_prob, y_test, "LogReg Top 20 genes")

In [ ]:
# Inkrementalny top-k (analogicznie do 4_pla2sig_gene_selection.py):
# dodajemy geny w kolejnosci rankingu SHAP, dla kazdego k liczymy CV AUC
# na train (mean +- std po foldach) i AUC na holdout test.
TOP_K_MAX     = 12
INCR_CV_FOLDS = 20
DELTA_AUC_SIG = 0.005

ranked_genes = ranking_df.head(TOP_K_MAX)['gene'].tolist()
X_tr_df = pd.DataFrame(X_train, columns=feature_names)
X_te_df = pd.DataFrame(X_test,  columns=feature_names)
y_tr_np = np.asarray(y_train)
y_te_np = np.asarray(y_test)

n_folds = min(INCR_CV_FOLDS, int(np.bincount(y_tr_np.astype(int)).min()))
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=2137)

rows = []
for k in range(1, len(ranked_genes) + 1):
    genes_k = ranked_genes[:k]
    Xtr_k = X_tr_df[genes_k].values
    Xte_k = X_te_df[genes_k].values

    aucs = []
    for tr, va in skf.split(Xtr_k, y_tr_np):
        m = LogisticRegression(
            solver='saga', max_iter=15000, class_weight='balanced',
            fit_intercept=True, random_state=2137,
        ).fit(Xtr_k[tr], y_tr_np[tr])
        aucs.append(roc_auc_score(y_tr_np[va], m.predict_proba(Xtr_k[va])[:, 1]))

    # refit na pelnym train -> AUC na holdout test
    m_full = LogisticRegression(
        solver='saga', max_iter=15000, class_weight='balanced',
        fit_intercept=True, random_state=2137,
    ).fit(Xtr_k, y_tr_np)
    auc_te = roc_auc_score(y_te_np, m_full.predict_proba(Xte_k)[:, 1])

    rows.append({
        'k':          k,
        'gene_added': genes_k[-1],
        'auc_mean':   float(np.mean(aucs)),
        'auc_std':    float(np.std(aucs, ddof=1)),
        'auc_test':   float(auc_te),
    })
    print(f"  top-{k:>2}  CV AUC = {rows[-1]['auc_mean']:.4f}+-{rows[-1]['auc_std']:.4f}"
          f"  | test AUC = {rows[-1]['auc_test']:.4f}  (+{genes_k[-1]})")

incr_df = pd.DataFrame(rows)

# k* = najmniejsze k ktorego CV AUC jest w odleglosci <= DELTA od max
auc_arr = incr_df['auc_mean'].values
k_star = int(incr_df['k'].values[auc_arr >= auc_arr.max() - DELTA_AUC_SIG].min())
minimal_genes = ranked_genes[:k_star]
print(f"\nk* = {k_star};  geny: {minimal_genes}")

# Wykres analogiczny do PLA2Sig: errorbar CV + linia test + vline k*
fig, ax = plt.subplots(figsize=(8, 4.5))

# clip wasow do [0,1] zeby nie wyjezdzaly powyzej 1.0
upper = np.minimum(incr_df['auc_mean'] + incr_df['auc_std'], 1.0)
lower = np.maximum(incr_df['auc_mean'] - incr_df['auc_std'], 0.0)
yerr  = np.vstack([incr_df['auc_mean'].values - lower.values,
                   upper.values - incr_df['auc_mean'].values])

ax.errorbar(incr_df['k'], incr_df['auc_mean'], yerr=yerr,
            marker='o', capsize=3,
            label=f'CV AUC ({n_folds}-fold, train)')
ax.plot(incr_df['k'], incr_df['auc_test'],
        marker='s', linestyle='--', color='tab:green',
        label='Holdout test AUC')
ax.set(xlabel='Top-k genow (wg stacking-weighted SHAP)',
       ylabel='AUC',
       title='Inkrementalny top-k: CV AUC (train) vs holdout test')
ax.set_xticks(incr_df['k'].values)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# best (jesli ktos chce wziac maksimum zamiast plateau k*)
best_row = incr_df.loc[incr_df['auc_test'].idxmax()]
print(f"\nMax test AUC: {best_row['auc_test']:.4f} przy k={int(best_row['k'])}")
print(f"Geny przy max: {ranked_genes[:int(best_row['k'])]}")
